# Exploratory data analysis: Telco Customer Churn

All cleaning goes through `clean_telco` in `src/features.py`; no cleaning code lives here.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import pandas as pd

from src.features import clean_telco

In [ ]:
raw = pd.read_csv(ROOT / "data/raw/telco_churn.csv")
df = clean_telco(raw)

In [ ]:
import matplotlib.pyplot as plt

## 1. Overview

In [ ]:
raw.shape

In [ ]:
raw.dtypes

In [ ]:
raw.head()

## 2. Blank TotalCharges

`TotalCharges` is read as text, not as a number, because some values are blank.

In [ ]:
raw["TotalCharges"].dtype

In [ ]:
blank = raw["TotalCharges"].str.strip() == ""
blank.sum()

The blank rows are exactly the customers with `tenure == 0`: brand-new customers with no bill yet.

In [ ]:
(blank == (raw["tenure"] == 0)).all()

In [ ]:
raw.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

## 3. Target balance

In [ ]:
df["Churn"].mean()

In [ ]:
ax = df["Churn"].value_counts().sort_index().plot(kind="bar")
ax.set_title("Target balance")
ax.set_xlabel("Churn (0 = No, 1 = Yes)")
ax.set_ylabel("Number of customers")
plt.show()

## 4. Churn rate by category

In [ ]:
def plot_churn_rate(by, title, xlabel):
    rate = df.groupby(by, observed=True)["Churn"].mean()
    ax = rate.plot(kind="bar")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Churn rate")
    plt.xticks(rotation=30, ha="right")
    plt.show()
    return rate

In [ ]:
plot_churn_rate("Contract", "Churn rate by contract type", "Contract")

In [ ]:
plot_churn_rate("InternetService", "Churn rate by internet service", "Internet service")

In [ ]:
plot_churn_rate("PaymentMethod", "Churn rate by payment method", "Payment method")

In [ ]:
tenure_bin = pd.cut(df["tenure"], [0, 12, 24, 48, 72])
plot_churn_rate(tenure_bin, "Churn rate by tenure", "Tenure (months)")

## 5. Numeric features by churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ["tenure", "MonthlyCharges"], strict=True):
    df.boxplot(column=col, by="Churn", ax=ax)
    ax.set_title(f"{col} by churn")
    ax.set_xlabel("Churn (0 = No, 1 = Yes)")
    ax.set_ylabel(col)
fig.suptitle("")
plt.show()

## 6. Key findings

- The target is imbalanced: about **26.5%** of customers churn, so accuracy alone is misleading
  (use ROC AUC / F1).
- `TotalCharges` has **11 blank values**, exactly the customers with `tenure == 0`; they are
  NaN after `clean_telco` and must be imputed on the train split only.
- **Contract** is the strongest signal: month-to-month customers churn at ~43%, versus ~11%
  (one year) and ~3% (two year).
- **Fiber optic** customers churn at ~42%, and **electronic check** payers at ~45%, far above
  the other groups.
- Churn falls with **tenure**: ~48% in the first 12 months versus ~10% after 48 months;
  churners have a median tenure of 10 months versus 38.
- Churners pay more per month (median `MonthlyCharges` ~80 versus ~64).